# Exploratory Data Analysis

This notebook looks at the sample qualifying, results, and driver standings files in `sample_data/`. The files are small, so the plots are mainly for getting familiar with the data.


In [ ]:
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt

qualifying = pd.read_csv(Path('sample_data') / 'qualifying.csv')
results_raw = pd.read_csv(Path('sample_data') / 'results.csv')
standings = pd.read_csv(Path('sample_data') / 'driver_standings.csv')
datasets = {'qualifying': qualifying, 'results': results_raw, 'driver_standings': standings}
print({name: df.shape for name, df in datasets.items()})


## 1. Profiles

For each file, we checked the shape, column types, missing values, and summary statistics.


In [ ]:
for name, df in datasets.items():
    print(f'\n--- {name}: {df.shape} ---')
    df.info()
    print('\nNull counts')
    print(df.isna().sum())
    print('\nSummary')
    print(df.describe(include='all').T)


The qualifying and results samples are both from race 18. The standings sample also has two rows from race 19. None of the CSV files has actual null values, but the results file uses `\N` as a placeholder.


## 2. Data checks

We checked for duplicates, placeholder values, and fields that need cleaning before use.


In [ ]:
print('Duplicate rows:', {name: int(df.duplicated().sum()) for name, df in datasets.items()})
print(r'\N counts in results:')
print(results_raw.eq(r'\N').sum()[lambda s: s > 0])

results = results_raw.replace(r'\N', pd.NA).copy()
results['milliseconds_numeric'] = pd.to_numeric(results['milliseconds'], errors='coerce')
joined = qualifying.merge(results, on=['raceId', 'driverId', 'constructorId'], suffixes=('_qual', '_result'))
standings_joined = results.merge(standings, on=['raceId', 'driverId'], suffixes=('_result', '_standing'))
joined['position_change'] = joined['position_qual'] - joined['positionOrder']

print('Rows with a placeholder:', results_raw.eq(r'\N').any(axis=1).sum())
print('Qualifying/results matches:', len(joined))
print('Results/standings matches:', len(standings_joined))
print('Grid different from qualifying:', (joined['grid'] != joined['position_qual']).sum())


There are no exact duplicate rows. Five results rows contain `\N` values. We keep the original data unchanged and only replace those values in the analysis copy. The race-time column also mixes a full time with time gaps, so we did not use it for calculations.


## 3. Distributions

These plots show the main fields we can explore with the sample.


In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(11, 8))
axes[0, 0].hist(qualifying['position'], bins=range(1, 12), align='left', edgecolor='white')
axes[0, 0].set(title='Qualifying position', xlabel='Position (1 = fastest)', ylabel='Count')
axes[0, 1].hist(results['positionOrder'], bins=range(1, 12), align='left', edgecolor='white')
axes[0, 1].set(title='Finishing position', xlabel='Position (1 = winner)', ylabel='Count')
axes[1, 0].hist(results['fastestLapSpeed'], bins=5, edgecolor='white')
axes[1, 0].set(title='Fastest-lap speed', xlabel='Speed (km/h)', ylabel='Count')
axes[1, 1].hist(joined['position_change'], bins=range(-8, 6), align='left', edgecolor='white')
axes[1, 1].set(title='Position change from qualifying', xlabel='Places gained (+) / lost (-)', ylabel='Count')
plt.tight_layout()
plt.show()
print(results[['points', 'fastestLapSpeed']].describe())


Qualifying and finishing positions each run from 1 to 10 once in this sample. Points range from 0 to 10. Fastest-lap speed ranges from about 213 to 218 km/h. Position change ranges from losing 7 places to gaining 4 places in the six matched rows.


## 4. Relationships

### Qualifying position and finishing position

This checks whether drivers who qualify near the front also finish near the front.


In [ ]:
fig, ax = plt.subplots(figsize=(6, 4))
ax.scatter(joined['position_qual'], joined['positionOrder'])
ax.plot([1, 10], [1, 10], '--', color='gray')
ax.invert_xaxis(); ax.invert_yaxis()
ax.set(xlabel='Qualifying position (1 = fastest)', ylabel='Finish position (1 = winner)')
plt.show()
print('Correlation:', round(joined['position_qual'].corr(joined['positionOrder']), 3))


The relationship is positive (0.347), but the sample is only six matched rows and some drivers move a lot. Next, we should add more races and separate finishers from retirements.


### Fastest-lap speed and finishing position

This looks at whether a driver's fastest lap is related to where they finish.


In [ ]:
fig, ax = plt.subplots(figsize=(6, 4))
ax.scatter(results['fastestLapSpeed'], results['positionOrder'])
ax.invert_yaxis()
ax.set(xlabel='Fastest-lap speed (km/h)', ylabel='Finish position (1 = winner)')
plt.show()
print('Correlation:', round(results['fastestLapSpeed'].corr(results['positionOrder']), 3))


Faster laps generally go with better finishing positions here (correlation −0.631). A fastest lap is only one lap, though. We should look at normal lap pace and race conditions next.


### Race results and driver standings

This uses the standings file as a second dataset.


In [ ]:
fig, ax = plt.subplots(figsize=(6, 4))
ax.scatter(standings_joined['positionOrder'], standings_joined['position_standing'])
ax.plot([1, 10], [1, 10], '--', color='gray')
ax.invert_xaxis(); ax.invert_yaxis()
ax.set(xlabel='Finish position (1 = winner)', ylabel='Standing position (1 = leader)')
plt.show()
print('Joined rows:', len(standings_joined))


The eight joined rows have the same race and standings rank. That makes sense for this small sample. With full standings data, we could look at how rankings change across several races.


## 5. OpenF1 API exploration

The sample CSVs only cover one race, so this section checks what OpenF1 offers for the broader questions (pit stops, weather, race control). It pulls data for 2023-2026 races and plots coverage and variation across races.


In [ ]:
import time, requests

BASE = "https://api.openf1.org/v1"

def get(endpoint, **params):
    for _ in range(4):
        r = requests.get(f"{BASE}/{endpoint}", params=params, timeout=60)
        if r.status_code == 429: time.sleep(10); continue
        time.sleep(2.1)                       # free tier: 30 requests/min
        if r.status_code == 404: return pd.DataFrame()
        r.raise_for_status()
        return pd.DataFrame(r.json())
    raise RuntimeError(endpoint)

sessions = pd.concat([get("sessions", year=y) for y in range(2023, 2027)])
races = sessions[(sessions.session_name == "Race") & (~sessions.is_cancelled)].sort_values("date_start")
races = races.assign(label=races.year.astype(str) + " " + races.circuit_short_name)
order = races.label

def pull(endpoint):
    dfs = [get(endpoint, session_key=r.session_key).assign(label=r.label) for _, r in races.iterrows()]
    return pd.concat([d for d in dfs if not d.empty])

pit, weather, rc = pull("pit"), pull("weather"), pull("race_control")
print(len(races), "races;", len(pit), "pit stops;", len(weather), "weather rows;", len(rc), "race control events")


In [ ]:
# 1. Sessions per season
sessions.pivot_table(index="year", columns="session_type", values="session_key",
                     aggfunc="count").plot.bar(stacked=True, title="Sessions per season")

# 2. Pit stop coverage per race
cov = pit.groupby("label").agg(stops=("lap_number", "size"),
                               has_stop=("stop_duration", lambda s: s.notna().mean())).reindex(order)
cov.plot.bar(subplots=True, figsize=(14, 6), legend=False, title=["pit stops", "share with stop_duration"])

# 3. Weather per race
weather["rain"] = weather.rainfall.astype(float) > 0
weather.groupby("label").agg(track_temp=("track_temperature", "mean"),
                             rain_share=("rain", "mean")).reindex(order).plot.bar(subplots=True, figsize=(14, 6))

# 4. Safety cars and red flags per race
pd.DataFrame({"safety_car": rc[rc.category == "SafetyCar"].groupby("label").size(),
              "red_flag": rc[rc.flag == "RED"].groupby("label").size()}
            ).reindex(order).fillna(0).plot.bar(stacked=True, figsize=(14, 4))
plt.show()


## Next questions

- How does qualifying position relate to finishing position with a full season of data?
- Which drivers gain or lose the most places from qualifying or the grid?
- Can we add pit-stop, circuit, weather, and lap data for the broader project questions?
